In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())
!nvidia-smi

True
Tesla T4
Tue Jul 14 19:19:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------------

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
print(os.listdir('/content/drive/MyDrive/dysarthria-data'))

['TORGO', 'UASpeech', 'MISTY']


In [4]:
!git clone https://github.com/paloma888/dysarthria-asr-personalization.git

Cloning into 'dysarthria-asr-personalization'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 111 (delta 47), reused 79 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 97.86 KiB | 894.00 KiB/s, done.
Resolving deltas: 100% (47/47), done.


In [5]:
!pip install transformers datasets peft accelerate evaluate jiwer soundfile librosa torchcodec -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.0 MB/s eta 0:00:00


In [6]:
import sys
sys.path.append('/content/dysarthria-asr-personalization/training')
from pipeline import processor, dataset_from_pairs, build_training_info, DataCollatorSpeechSeq2Seq

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

In [7]:
!cp -r /content/drive/MyDrive/dysarthria-data/MISTY /content/data

In [8]:
import csv
from pathlib import Path

def read_csv_prompts(folder):
  train_iso, train_cont, val, test_a, test_b = [], [], [], [], []
  with open('/content/dysarthria-asr-personalization/dataset_design/word_prompts.txt', mode='r', encoding='utf-8') as file:
    dict_reader = csv.DictReader(file)
    for row in dict_reader:
      wavpath = Path(folder) / "audio" / row["filename"]
      if not wavpath.exists():
        print(f"doesn't exist: {wavpath}")
      is_isolated = (len(row["text"].split()) == 1)
      wav_prompt_pair = {"audio": str(wavpath), "text": row["text"], "is_isolated": is_isolated}
      if row["split"] == "train":
        if is_isolated:
          train_iso.append(wav_prompt_pair)
        else:
          train_cont.append(wav_prompt_pair)
      else:
        if row["eval_set"].strip() == "A":
          test_a.append(wav_prompt_pair)
        elif row["eval_set"].strip() == "B":
          test_b.append(wav_prompt_pair)

  return train_iso, train_cont, val, test_a, test_b



train_iso, train_cont, val, test_a, test_b = read_csv_prompts("/content/data")
print(f"Train iso: {len(train_iso)}, Train cont: {len(train_cont)}, Set A: {len(test_a)}, Set B: {len(test_b)}")

Train iso: 99, Train cont: 26, Set A: 56, Set B: 12


In [9]:
import random
random.seed(20)
random.shuffle(train_iso)
random.shuffle(train_cont)

subset_iso = len(train_iso) // 8
subset_cont = len(train_cont) // 8

val = train_iso[:subset_iso] + train_cont[:subset_cont]
train = train_iso[subset_iso:] + train_cont[subset_cont:]

In [10]:
random.shuffle(train)
random.shuffle(test_a)
random.shuffle(test_b)
random.shuffle(val)

print(f"Train: {len(train)}, Val: {len(val)}, Set A: {len(test_a)}, Set B: {len(test_b)}")

Train: 110, Val: 15, Set A: 56, Set B: 12


In [11]:
train_ds = dataset_from_pairs(train)
val_ds = dataset_from_pairs(val)
test_a_ds = dataset_from_pairs(test_a)
test_b_ds = dataset_from_pairs(test_b)


train_ds = train_ds.map(build_training_info)
val_ds = val_ds.map(build_training_info)

import numpy as np
print(np.shape(train_ds[0]["input_features"]))

total_iso_test = sum(1 for p in test_a if p["is_isolated"])
total_cont_test = sum(1 for p in test_a if not p["is_isolated"])

print("iso in set A: ", total_iso_test)
print("cont in set A: ", total_cont_test)

Map:   0%|          | 0/110 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

(80, 3000)
iso in set A:  21
cont in set A:  35


In [12]:
!pip install -U torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 47.0 MB/s eta 0:00:00


In [13]:
!pip install -U hf_transfer -q
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 16.9 MB/s eta 0:00:00


In [14]:
from transformers import WhisperForConditionalGeneration
from peft import get_peft_model, LoraConfig, PeftModel

base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").to("cuda")
base.config.forced_decoder_ids = None
base.config.suppress_tokens = []

base = PeftModel.from_pretrained(base, "/content/drive/MyDrive/clearly-base-adapters/adapter-dropout")
base = base.merge_and_unload()

lora = LoraConfig(r = 8, lora_alpha = 16, target_modules = ["q_proj", "v_proj"], lora_dropout = 0.05)
model = get_peft_model(base, lora)
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

trainable params: 884,736 || all params: 242,619,648 || trainable%: 0.3647


In [15]:
from pipeline import processor
from pipeline import DataCollatorSpeechSeq2Seq
collator = DataCollatorSpeechSeq2Seq(processor)

In [16]:
from transformers import Seq2SeqTrainingArguments, EarlyStoppingCallback
training_args = Seq2SeqTrainingArguments(
    output_dir = "/content/drive/MyDrive/checkpoints-personalize-1",
    per_device_train_batch_size=4,
    learning_rate=1e-4,
    num_train_epochs=8,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
    seed=20,
    load_best_model_at_end = True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    warmup_steps = 10,
)

In [17]:
import transformers
transformers.logging.set_verbosity_error()

In [19]:
from transformers import Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model = model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    processing_class= processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.500077,2.326219
2,1.791354,1.870524
3,1.658581,1.610789
4,1.196831,1.536985
5,1.496677,1.493576
6,1.032759,1.465608
7,1.238739,1.461160
8,1.120054,1.457639


TrainOutput(global_step=224, training_loss=1.5569435240966933, metrics={'train_runtime': 277.271, 'train_samples_per_second': 3.174, 'train_steps_per_second': 0.808, 'total_flos': 2.550762897408e+17, 'train_loss': 1.5569435240966933, 'epoch': 8.0})

In [21]:
device = "cuda"
model.to(device)

PeftModel(
  (base_model): LoraModel(
    (model): WhisperForConditionalGeneration(
      (model): WhisperModel(
        (encoder): WhisperEncoder(
          (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
          (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
          (embed_positions): Embedding(1500, 768)
          (layers): ModuleList(
            (0-11): 12 x WhisperEncoderLayer(
              (self_attn): WhisperAttention(
                (k_proj): Linear(in_features=768, out_features=768, bias=False)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
                  (lora_B): ModuleDict(
   

In [22]:
from eval import inference, get_metrics
with model.disable_adapter():
    base_a_wer_cont, base_a_cer_cont, base_a_wra_iso, base_a_cer_iso = get_metrics(model, test_a_ds)
    base_b_wer_cont, base_b_cer_cont, base_b_wra_iso, base_b_cer_iso = get_metrics(model, test_b_ds)

print(f"Set A Baseline Continuous: WER: {base_a_wer_cont:.2%}, CER: {base_a_cer_cont:.2%}")
print(f"Set A Baseline Isolated: CER: {base_a_cer_iso:.2%}, WRA: {base_a_wra_iso:.2%}")
print(f"Set B Baseline Continuous: WER: {base_b_wer_cont:.2%}, CER: {base_b_cer_cont:.2%}")
print(f"Set B Baseline Isolated: CER: {base_b_cer_iso:.2%}, WRA: {base_b_wra_iso:.2%}")

Set A Baseline Continuous: WER: 96.67%, CER: 82.65%
Set A Baseline Isolated: CER: 78.05%, WRA: 9.52%
Set B Baseline Continuous: WER: 100.00%, CER: 87.65%
Set B Baseline Isolated: CER: 69.57%, WRA: 0.00%


In [30]:
#one set A example to view
pred = inference(model, test_a_ds[4]["audio"], device)
print("Predicted:", pred)
print("Actual:", test_a_ds[4]["text"])
print(test_a_ds[0]["audio"]["sampling_rate"])

Predicted: little
Actual: open the window
16000


In [24]:
a_wer_cont, a_cer_cont, a_wra_iso, a_cer_iso = get_metrics(model, test_a_ds)
b_wer_cont, b_cer_cont, b_wra_iso, b_cer_iso = get_metrics(model, test_b_ds)

print(f"Set A FT Continuous: WER: {a_wer_cont:.2%}, CER: {a_cer_cont:.2%}")
print(f"Set A FT Isolated: CER: {a_cer_iso:.2%}, WRA: {a_wra_iso:.2%}")
print(f"Set B FT Continuous: WER: {b_wer_cont:.2%}, CER: {b_cer_cont:.2%}")
print(f"Set B FT Isolated: CER: {b_cer_iso:.2%}, WRA: {b_wra_iso:.2%}")

for row in test_b_ds:
  pred = inference(model, row["audio"], device)
  print(f"[{'iso' if row['is_isolated'] else 'context'}] pred: {pred!r}  actual: {row['text']!r}")

Set A FT Continuous: WER: 91.11%, CER: 75.66%
Set A FT Isolated: CER: 76.83%, WRA: 4.76%
Set B FT Continuous: WER: 100.00%, CER: 81.48%
Set B FT Isolated: CER: 52.17%, WRA: 16.67%
[context] pred: 'pagan'  actual: 'pet the cat'
[context] pred: 'ploughshare'  actual: 'watch the game'
[context] pred: 'mike it'  actual: 'put on the cap'
[context] pred: "what's up"  actual: 'wash the clothes'
[context] pred: 'people'  actual: 'ring the bell'
[iso] pred: 'can'  actual: 'cat'
[iso] pred: 'cat'  actual: 'cap'
[context] pred: 'nodule'  actual: 'read the book'
[iso] pred: 'what'  actual: 'watch'
[iso] pred: 'read'  actual: 'read'
[iso] pred: 'weed'  actual: 'ring'
[iso] pred: 'what'  actual: 'wash'


In [25]:
model.save_pretrained("/content/drive/MyDrive/clearly-base-adapters/adapter-personalize")
import json
with open("/content/drive/MyDrive/log_history_personalize-1.json","w") as f:
    json.dump(trainer.state.log_history, f, allow_nan=False, default=str)